# 05 — Analyse de rentabilisation

Traduire les erreurs du modèle en coûts et comparer trois stratégies :

| Stratégie | Règle |
|---|---|
| Intervalle fixe | maintenance au même cycle pour tous les moteurs (intervalle optimal a posteriori) |
| Prédictive | maintenance quand la RUL prédite passe sous un seuil (+ délai d'intervention) |
| Idéale | connaissance parfaite du cycle de panne (borne inférieure) |

⚠️ **Les montants sont des hypothèses de travail**, pas des chiffres réels : ils sont documentés ci-dessous et soumis à une analyse de sensibilité.

La simulation utilise les moteurs d'**entraînement** (dont on connaît le cycle de panne) avec des prédictions **hors échantillon** (`GroupKFold` par moteur) à chaque cycle.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SUBSET = "FD001"   # puis "FD004"


In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from xgboost import XGBRegressor

from src.data import prepare, load_subset
from src.features import rolling_features, feature_columns
from src.cost_model import CostAssumptions, compare_strategies, threshold_sweep, sensitivity

CAP = 125
train, _, _, sensors = prepare(SUBSET, cap=CAP)
train_f = rolling_features(train, sensors, 10)
cols = feature_columns(sensors)

# Prédictions hors échantillon : chaque moteur est prédit par un modèle qui ne l'a jamais vu.
# (On peut remplacer XGBoost par le LSTM du notebook 04 avec la même logique.)
model = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                     colsample_bytree=0.8, random_state=0)
oof = cross_val_predict(model, train_f[cols], train_f["RUL"], groups=train_f["unit"], cv=GroupKFold(5))

raw, _, _ = load_subset(SUBSET)
failure = raw.groupby("unit")["cycle"].max()
preds = pd.DataFrame({"unit": train_f["unit"], "cycle": train_f["cycle"], "pred": oof})
preds["failure_cycle"] = preds["unit"].map(failure)
preds.head()

## Hypothèses de coûts (à documenter dans le rapport)

In [ ]:
a = CostAssumptions(
    planned_maintenance=50_000,   # $ par maintenance planifiée — hypothèse
    unplanned_failure=500_000,    # $ par panne imprévue (réparation + avion immobilisé, AOG) — hypothèse
    value_per_cycle=1_000,        # $ de vie utile perdue par cycle de remplacement précoce — hypothèse
    lead_time=5,                  # cycles entre l'alerte et l'intervention
)
pd.Series(a.as_dict(), name="valeur")

In [ ]:
table = compare_strategies(preds, a)
table.to_csv(ROOT / "reports" / f"strategies_{SUBSET}.csv", index=False)
table.round(1)

In [ ]:
sweep = threshold_sweep(preds, a)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sweep["paramètre"], sweep["coût par moteur"] / 1000, "o-", label="coût par moteur")
ax1.axhline(table.loc[0, "coût par moteur"] / 1000, color="grey", ls="--", label="intervalle fixe optimal")
ax1.axhline(table.loc[2, "coût par moteur"] / 1000, color="green", ls=":", label="idéale")
ax1.set(xlabel="seuil de RUL prédite déclenchant la maintenance", ylabel="coût par moteur (k$)")
ax2 = ax1.twinx(); ax2.bar(sweep["paramètre"], sweep["pannes"], width=3, alpha=0.2, color="red")
ax2.set_ylabel("pannes imprévues", color="red")
ax1.legend(loc="upper right"); ax1.set_title("Choix du seuil de maintenance")
plt.tight_layout(); plt.savefig(FIG / f"seuil_cout_{SUBSET}.png", dpi=150); plt.show()

## Analyse de sensibilité

L'économie dépend fortement du rapport entre le coût d'une panne et la valeur d'un cycle : on vérifie que la conclusion tient sur une plage d'hypothèses raisonnables.

In [ ]:
sens = sensitivity(preds, a)
sens.pivot(index="coût panne", columns="valeur d'un cycle", values="économie par moteur").round(0)

## Message pour la direction (à reprendre dans le résumé d'une page)

- Avec les hypothèses de base, la maintenance prédictive économise **[X] $ par moteur** par rapport au meilleur intervalle fixe, avec **[n] pannes imprévues** au lieu de **[m]**.
- Seuil recommandé : planifier la maintenance quand la RUL prédite passe sous **[s] cycles**.
- Conclusion robuste / non robuste aux hypothèses de coûts (voir tableau de sensibilité).
- Limites : données simulées ; en exploitation réelle, capteurs manquants, flottes hétérogènes et historique de maintenance incomplet.